# Anomaly Detection Training


This notebook trains anomaly detection models to identify unusual electricity demand patterns.

## Methods Used:
- **Isolation Forest**: Unsupervised anomaly detection with contamination=0.03
- **Z-Score**: Statistical anomaly detection using threshold=3.0
- **Combined Method**: OR logic combining both methods

## Steps:
1. Load daily profiles and clustering results
2. Load profile_scaler.pkl and scale features
3. Train Isolation Forest and save to models/isolation_forest_model.pkl
4. Calculate Z-Score anomalies
5. Combine methods and create anomaly reasons
6. Save anomaly results and metrics


In [ ]:

import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
import sys
from scipy import stats

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import sklearn components
from sklearn.ensemble import IsolationForest

# Set up paths
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
DAILY_PROFILES_PATH = DATA_PROCESSED_DIR / "daily_profiles.csv"
CLUSTERING_RESULTS_PATH = DATA_PROCESSED_DIR / "clustering_results.csv"
ANOMALY_RESULTS_PATH = DATA_PROCESSED_DIR / "anomaly_results.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Daily profiles path: {DAILY_PROFILES_PATH}")
print(f"Clustering results path: {CLUSTERING_RESULTS_PATH}")


In [ ]:

# Load daily profiles
if not DAILY_PROFILES_PATH.exists():
    raise FileNotFoundError(f"Daily profiles not found at {DAILY_PROFILES_PATH}")

daily_profiles = pd.read_csv(DAILY_PROFILES_PATH)
print(f"Daily profiles loaded: {daily_profiles.shape}")

# Load clustering results
if not CLUSTERING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Clustering results not found at {CLUSTERING_RESULTS_PATH}")

clustering_results = pd.read_csv(CLUSTERING_RESULTS_PATH)
print(f"Clustering results loaded: {clustering_results.shape}")

# Load profile scaler
scaler_path = MODELS_DIR / "profile_scaler.pkl"
if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler not found at {scaler_path}")

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)
print(f"Scaler loaded successfully")


In [ ]:

# Prepare features for anomaly detection
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]
available_cols = [col for col in profile_cols if col in daily_profiles.columns]
X = daily_profiles[available_cols].values

print(f"Features prepared:")
print(f"- Number of samples (days): {X.shape[0]}")
print(f"- Number of features (time slots): {X.shape[1]}")

# Scale features using the same scaler from clustering
X_scaled = scaler.transform(X)

print(f"Features standardized using existing scaler")
print(f"Scaled data shape: {X_scaled.shape}")


In [ ]:

# Train Isolation Forest model
print("Training Isolation Forest model...")

iso_forest = IsolationForest(
    n_estimators=300,
    contamination=0.03,  # Expect 3% anomalies
    random_state=42
)

iso_forest.fit(X_scaled)

# Calculate anomaly scores and predictions
iso_anomaly_scores = -iso_forest.decision_function(X_scaled)  # Higher = more anomalous
iso_anomaly_flags = iso_forest.predict(X_scaled) == -1  # -1 indicates anomaly

print(f"Isolation Forest trained successfully!")
print(f"Anomalies detected: {iso_anomaly_flags.sum()} ({iso_anomaly_flags.mean()*100:.2f}% of data)")
print(f"Expected contamination: 3.0%")
print(f"Actual contamination: {iso_anomaly_flags.mean()*100:.2f}%")

# Save Isolation Forest model
with open(MODELS_DIR / "isolation_forest_model.pkl", "wb") as f:
    pickle.dump(iso_forest, f)

print(f"Isolation Forest model saved to: {MODELS_DIR / 'isolation_forest_model.pkl'}")


In [ ]:

# Calculate Z-Score based anomaly detection
print("Calculating Z-Score based anomaly detection...")

# Calculate Z-Scores for daily statistics
peak_z_scores = np.abs(stats.zscore(daily_profiles['daily_peak'], nan_policy='omit'))
mean_z_scores = np.abs(stats.zscore(daily_profiles['daily_mean'], nan_policy='omit'))

# Define anomaly thresholds
Z_SCORE_THRESHOLD = 3.0

# Create anomaly flags
peak_z_anomaly = peak_z_scores >= Z_SCORE_THRESHOLD
mean_z_anomaly = mean_z_scores >= Z_SCORE_THRESHOLD

# Combined Z-score anomaly flag
z_score_anomaly_flags = peak_z_anomaly | mean_z_anomaly

print(f"Z-Score anomaly detection completed!")
print(f"Peak demand anomalies: {peak_z_anomaly.sum()} ({peak_z_anomaly.mean()*100:.2f}%)")
print(f"Mean demand anomalies: {mean_z_anomaly.sum()} ({mean_z_anomaly.mean()*100:.2f}%)")
print(f"Combined Z-score anomalies: {z_score_anomaly_flags.sum()} ({z_score_anomaly_flags.mean()*100:.2f}%)")


In [ ]:

# Combine anomaly detection methods
print("Combining multiple anomaly detection methods...")

# Combine anomaly flags using OR logic
final_anomaly_flags = iso_anomaly_flags | z_score_anomaly_flags

print(f"Combined anomaly detection results:")
print(f"Isolation Forest anomalies: {iso_anomaly_flags.sum()} ({iso_anomaly_flags.mean()*100:.2f}%)")
print(f"Z-Score anomalies: {z_score_anomaly_flags.sum()} ({z_score_anomaly_flags.mean()*100:.2f}%)")
print(f"Final combined anomalies: {final_anomaly_flags.sum()} ({final_anomaly_flags.mean()*100:.2f}%)")

# Create anomaly reasons
anomaly_reasons = []
for i in range(len(daily_profiles)):
    reasons = []
    if iso_anomaly_flags[i]:
        reasons.append(f"Isolation Forest (score: {iso_anomaly_scores[i]:.3f})")
    if peak_z_anomaly[i]:
        reasons.append(f"Peak Z-Score ({peak_z_scores[i]:.2f})")
    if mean_z_anomaly[i]:
        reasons.append(f"Mean Z-Score ({mean_z_scores[i]:.2f})")
    anomaly_reasons.append("; ".join(reasons) if reasons else "No anomaly")

print(f"Anomaly reasons generated for all days.")


In [ ]:

# Prepare anomaly results dataframe
anomaly_results = pd.DataFrame({
    'date': daily_profiles['date'],
    'isolation_anomaly_score': iso_anomaly_scores,
    'isolation_anomaly_flag': iso_anomaly_flags,
    'peak_z_score': peak_z_scores,
    'mean_z_score': mean_z_scores,
    'z_score_anomaly_flag': z_score_anomaly_flags,
    'final_anomaly_flag': final_anomaly_flags,
    'anomaly_reason': anomaly_reasons
})

# Merge with clustering results
anomaly_results = anomaly_results.merge(
    clustering_results[['date', 'kmeans_cluster', 'dbscan_cluster', 'dbscan_noise_flag']],
    on='date',
    how='left'
)

# Add daily statistics for analysis
anomaly_results = anomaly_results.merge(
    daily_profiles[['date', 'daily_peak', 'daily_mean', 'daily_std']],
    on='date',
    how='left'
)

print(f"Anomaly results prepared: {anomaly_results.shape}")
print(f"\nSample of anomaly results:")
print(anomaly_results[['date', 'final_anomaly_flag', 'isolation_anomaly_score', 
                      'peak_z_score', 'mean_z_score', 'anomaly_reason']].head())


In [ ]:

# Save anomaly results
anomaly_results.to_csv(ANOMALY_RESULTS_PATH, index=False)
print(f"Anomaly results saved to: {ANOMALY_RESULTS_PATH}")

# Calculate and save anomaly metrics
total_days = len(anomaly_results)
anomaly_metrics = {
    'total_days': total_days,
    'anomaly_days': int(final_anomaly_flags.sum()),
    'anomaly_percentage': round(float(final_anomaly_flags.mean()) * 100, 2),
    'isolation_forest_anomaly_count': int(iso_anomaly_flags.sum()),
    'z_score_anomaly_count': int(z_score_anomaly_flags.sum()),
    'final_anomaly_count': int(final_anomaly_flags.sum()),
    'isolation_forest_metrics': {
        'contamination_setting': 0.03,
        'actual_contamination': round(float(iso_anomaly_flags.mean()), 4),
        'n_estimators': 300
    },
    'z_score_metrics': {
        'threshold': Z_SCORE_THRESHOLD,
        'peak_z_anomalies': int(peak_z_anomaly.sum()),
        'mean_z_anomalies': int(mean_z_anomaly.sum())
    },
    'data_info': {
        'date_range_start': daily_profiles['date'].min(),
        'date_range_end': daily_profiles['date'].max()
    }
}

# Save metrics
with open(MODELS_DIR / "anomaly_metrics.json", 'w') as f:
    json.dump(anomaly_metrics, f, indent=2)

print(f"Anomaly metrics saved to: {MODELS_DIR / 'anomaly_metrics.json'}")

print("\n=== ANOMALY DETECTION SUMMARY ===")
print(f"Total days analyzed: {total_days}")
print(f"Final anomalies detected: {anomaly_metrics['anomaly_days']} ({anomaly_metrics['anomaly_percentage']}%)")
print(f"Isolation Forest anomalies: {anomaly_metrics['isolation_forest_anomaly_count']}")
print(f"Z-Score anomalies: {anomaly_metrics['z_score_anomaly_count']}")
print(f"\nSaved files:")
print(f"- {MODELS_DIR / 'isolation_forest_model.pkl'}")
print(f"- {ANOMALY_RESULTS_PATH}")
print(f"- {MODELS_DIR / 'anomaly_metrics.json'}")
